## Read silver config File

In [0]:

import sys
sys.path.append("/Workspace/Users/nalluriravi3@gmail.com/Medalion_Project")

from utils.audit_manager import AuditManager
from utils.config_reader import ConfigReader
from utils.quarantine_manager import QuarantineManager
from pyspark.sql.functions import lit, when
from pyspark.sql.functions import current_timestamp, col,row_number
from datetime import datetime
from pyspark.sql.window import Window


config = ConfigReader.read_config("/Workspace/Users/nalluriravi3@gmail.com/Medalion_Project/Configs/silver_config.json")

silver_config = config["customer"]

target_df = spark.table(silver_config["target_table"])




## Read Mandatory columns and format the name

In [0]:
from pyspark.sql.functions import col, trim, initcap

bronze_df = spark.table(silver_config["source_table"])

silver_df = bronze_df.withColumn(
    "customer_name",
    trim(initcap(col(silver_config["mandatory_columns"][1])))
)



## Quarantine invalid records

In [0]:

from pyspark.sql.functions import current_timestamp, col,row_number
#cast salary column data type to int
silver_df = silver_df.withColumn("salary", col("salary").cast("int"))

# Valid records
valid_df = silver_df.filter(
    (col(silver_config["mandatory_columns"][0]).isNotNull()) &
    (col(silver_config["mandatory_columns"][1]).isNotNull()) &
    (col("contact").isNotNull()) &
    (col("contact").rlike("^[0-9]{10}$")) &
    (col("email").isNotNull()) &
    (col("email").rlike("^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")) &
    (col("salary").isNotNull()) &
    (col("salary") > 0)
)

#filter duplicate ids from source take only latest ingested record
valid_df=valid_df.withColumn(
    "rn",row_number().over(
        Window.partitionBy(col(silver_config["mandatory_columns"][0]))\
        .orderBy(col("ingestion_timestamp").desc())
    )
).filter(col("rn")==lit(1)).drop("rn")

# add scd2 columns
# valid_df=valid_df.withColumn(
#     "start_date",current_timestamp()
# ).withColumn(
#     "end_date",lit(None).cast("timestamp")
# ).withColumn(
#     "is_current",lit("Y")
# )
       

# Invalid records
invalid_df = silver_df.filter(
    (col("customer_id").isNull()) |
    (col("customer_name").isNull()) |
    (col("email").isNull()) |
    (~col("email").rlike("^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")) |
    (col("contact").isNull()) |
    (~col("contact").rlike("^[0-9]{10}$")) |
    (col("salary").isNull()) |
    (col("salary") <= 0)
)

# Rejected reason column
invalid_df = invalid_df.withColumn(
    "rejected_reason",
    when(col("customer_id").isNull(), lit("CUSTOMER_ID_NULL"))
    .when(col("customer_name").isNull(), lit("CUSTOMER_NAME_NULL"))
    .when(col("email").isNull(), lit("EMAIL_NULL"))
    .when(
        ~col("email").rlike(
            "^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        ),
        lit("INVALID_EMAIL")
    )
    .when(col("contact").isNull(), lit("CONTACT_NO_NULL"))
    .when(
        ~col("contact").rlike("^[0-9]{10}$"),
        lit("INVALID_CONTACT_NO")
    )
    .when(col("salary").isNull(), lit("SALARY_NULL"))
    .when(col("salary") <= 0, lit("INVALID_SALARY"))
).withColumn("rejected_timestamp", current_timestamp())

#drop unwated columns
invalid_df = invalid_df.drop("ingestion_timestamp","country","state","city")


# #write into quarantinie table
QuarantineManager.write_quarantine(
    invalid_df, silver_config["quarantine_table"]    
)

## Implement SCD2

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number,lit,col

# handle temp tables
spark.sql("""DROP TABLE IF EXISTS temp_changed_src""")
spark.sql("""DROP TABLE IF EXISTS temp_new_records""")

#identify changed records
changed_src_df=valid_df.alias("src").join(
    target_df.filter(col("is_current")==lit("Y")).alias("target"),col(
        "src.customer_id")==col("target.customer_id"),"inner")\
    .filter(
        (col("target.salary")!=col("src.salary")) &
        (col("target.is_current")==lit("Y"))
    )\
    .select(
        col("src.*")
    )

# filter dupliate records
changed_src_df=changed_src_df.withColumn(
    "rn",row_number().over(Window.partitionBy(
        "customer_id").orderBy(col(
        "ingestion_timestamp").desc()))).filter(col("rn")==1).drop("rn")

changed_src_df.write.format("delta")\
    .mode("overWrite")\
    .saveAsTable("temp_changed_src")

changed_rec_count=changed_src_df.count()
changed_src_df=spark.table("temp_changed_src")


#identify new records
new_records_df=valid_df.alias("src").join(
    target_df.alias("target"),
    col("src.customer_id")==col("target.customer_id"),"leftanti")


# filter dupliate records
new_records_df=new_records_df.withColumn(
    "rn",row_number().over(Window.partitionBy(
        "customer_id").orderBy(col("ingestion_timestamp").desc())))\
        .filter(col("rn")==1).drop("rn")

new_records_df.write.format("delta")\
    .mode("overWrite")\
    .saveAsTable("temp_new_records")

new_records_df=spark.table("temp_new_records")

# expire changed records in target table
if changed_rec_count > 0:
    delta_obj=DeltaTable.forName(spark,silver_config["target_table"])

    delta_obj.alias("target").merge(
        changed_src_df.alias("src"),
        col("target.customer_id")==col("src.customer_id")
    ).whenMatchedUpdate(
        set={
        "end_date":current_timestamp(),
        "is_current":lit("N")
    }).execute()
else:
    print("no changed records to expire")


# find final set of records to update into target table
final_df=changed_src_df.unionByName(new_records_df)

# Add scd columns update is_current and start_date
final_df=final_df.withColumn(
    "start_date", current_timestamp()
).withColumn(
    "end_date", lit(None).cast("timestamp")
).withColumn(
    "is_current", lit("Y")
)
# write into target table
if final_df.count() > 0:
    final_df.write.format("delta")\
            .option("mergeSchema", "true")\
            .mode("append") \
            .saveAsTable(silver_config["target_table"])
else:
    print("no new records to write")

spark.sql("""DROP TABLE IF EXISTS temp_changed_src""")
spark.sql("""DROP TABLE IF EXISTS temp_new_records""")